# Lab 07 — Neural Net from Scratch — MNIST
**Deep Learning Track** · Intermediate–Advanced · ~75 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Build a feed-forward MLP with numpy only
2. Implement forward pass, ReLU, softmax, cross-entropy
3. Train with one epoch loop and plot accuracy curve
4. Compare learning rates and inspect confusion

## Datasets (this folder)
- `mnist_train.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-07-neural-net-mnist/lab-07-neural-net-mnist.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`mnist_train.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-07-neural-net-mnist"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-07-neural-net-mnist/bundle/dataset.zip"
NEED = ["mnist_train.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Computer Vision: Handwritten Digit Recognition with a Neural Network from Scratch

> **Scenario:** You are building the core engine for a postal service that must automatically read handwritten ZIP codes on envelopes. Before reaching for a deep-learning framework, your team wants to understand exactly what happens inside a neural network — the forward pass, the loss, and the gradient updates — by building one from scratch with numpy only.
>
> **You will learn:** perceptron → MLP intuition, forward pass with matrix multiplication, ReLU and softmax activations, cross-entropy loss, one full backpropagation training epoch, learning-rate effects, and reading a confusion matrix.
> **Time:** ~75 minutes. **Level:** Intermediate–Advanced. **Needs:** pandas + numpy + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | Where it appears | Why it matters |
|---|---|---|
| Perceptron → MLP | Section 1 | One neuron can't separate digits; layers can |
| Forward pass | Section 1 | Matrix multiply + activation = prediction |
| ReLU / softmax | Section 1 | Non-linearity + probability distribution |
| Cross-entropy loss | Section 2 | How wrong are we, numerically? |
| Backpropagation | Section 2 | Chain rule assigns blame to each weight |
| Learning rate | Exercise 2 | Too big diverges, too small crawls |
| Confusion matrix | Exercise 3 | Which digits get mixed up? |

---

## 1. From Perceptron to MLP: The Forward Pass

Every time you unlock your phone with your face, scan a document into searchable text, or let your bank flag a suspicious transaction, a neural network is doing the recognizing. But before these systems can recognize anything, they have to *learn* — and learning starts with understanding the forward pass: the exact computation that turns a pile of pixel numbers into a prediction. This section builds that computation by hand, so that when you later call `model.fit()` in a framework, you know precisely what is happening under the hood.

A **perceptron** is a single neuron: it computes `output = activation(w · x + b)`. Picture it as a voting panel: each input pixel casts a vote weighted by how strongly the neuron cares about it, the bias sets the panel's overall enthusiasm, and the activation decides whether the neuron "fires." A single perceptron can only draw one straight line through data — it fails on anything non-linearly separable (like distinguishing handwritten 3s from 8s). The fix is a **multi-layer perceptron (MLP)**: stack neurons into layers so the network can carve the input space into complex regions.

Think of the hidden layer as a team of 64 feature detectors. The first layer's weights learn to respond to simple patterns — edges, corners, loops of ink. The output layer then votes on which digit those features collectively suggest. No single neuron "knows" what a 7 is; the knowledge is distributed across the weights, which is exactly what makes neural networks both powerful and hard to interpret.

Our architecture for MNIST:

```
Input (784) → Dense(64, ReLU) → Dense(10, softmax)
```

- **784 inputs** = one per pixel in a 28×28 grayscale image.
- **64 hidden neurons** with **ReLU** (`max(0, z)`) — the non-linearity that makes deep networks work. Without a non-linear activation, stacking layers is pointless: two linear collapses are still one linear collapse. ReLU passes positive signals through unchanged and zeros out negative ones, which keeps gradients flowing during training. Sigmoid, the older alternative, squashes everything into (0, 1) and its gradient shrinks toward zero for large inputs — the "vanishing gradient" problem that used to stall deep networks.
- **10 outputs** with **softmax** — converts raw scores into a probability distribution over digits 0–9.

The forward pass is just matrix multiplication plus activation, applied layer by layer. The code below loads 1,000 MNIST images, initializes small random weights, and runs one forward pass — no training yet, just the raw machinery:

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load the first 1000 rows — enough to train in seconds on CPU
df = pd.read_csv("mnist_train.csv", nrows=1000)
X = df.iloc[:, 1:].to_numpy(dtype=float) / 255.0   # pixels → [0, 1]
y = df["label"].to_numpy()                          # digit labels 0–9

# One-hot encode labels: digit 3 → [0,0,0,1,0,0,0,0,0,0]
Y = np.eye(10)[y]

# Initialize weights (small random values) and biases (zeros)
rng = np.random.default_rng(42)
W1 = rng.normal(0, 0.1, size=(784, 64))
b1 = np.zeros(64)
W2 = rng.normal(0, 0.1, size=(64, 10))
b2 = np.zeros(10)

def relu(z):
    return np.maximum(z, 0)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)   # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# Forward pass
z1 = X @ W1 + b1        # (1000, 64)  — hidden pre-activation
a1 = relu(z1)           # (1000, 64)  — hidden activation
z2 = a1 @ W2 + b2       # (1000, 10)  — output pre-activation
probs = softmax(z2)     # (1000, 10)  — class probabilities

predicted = probs.argmax(axis=1)
accuracy = (predicted == y).mean()
print(f"Untrained accuracy: {accuracy:.4f}  (random guessing ≈ 0.10)")
print(f"Probability row sums to 1: {np.allclose(probs.sum(axis=1), 1.0)}")
print(f"Sample prediction for row 0: digit {predicted[0]}, true label {y[0]}")


**What to notice:**
1. Untrained accuracy is ~0.06–0.10, close to random guessing (1/10) — the network knows nothing yet.
2. Each row of `probs` sums to exactly 1.0 — softmax guarantees a valid probability distribution.
3. `X @ W1` is a single matrix multiply: (1000×784) @ (784×64) → (1000×64). The whole batch is processed at once.

> **Pitfall:** Forgetting to scale pixels to [0, 1]. Raw pixel values (0–255) make the initial activations huge, softmax saturates, and gradients vanish — the network will barely learn.

---

## 2. Training: Cross-Entropy Loss and Backpropagation

**Cross-entropy loss** measures how far the predicted probability distribution is from the true one. For a single example with true class `c`: `loss = -log(p_c)`. Averaged over the batch:

```
loss = -mean( log( probs[range(N), y] ) )
```

**Backpropagation** applies the chain rule to compute how much each weight contributed to the error, then gradient descent nudges each weight opposite to its gradient:

```
W ← W - lr * dW
```

One complete training epoch = forward pass → loss → backward pass → weight update:

In [ ]:
def train_epoch(X, Y, W1, b1, W2, b2, lr):
    N = len(X)
    # --- Forward ---
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    probs = softmax(z2)

    # --- Loss ---
    loss = -np.log(probs[np.arange(N), Y.argmax(axis=1)] + 1e-12).mean()

    # --- Backward ---
    dz2 = (probs - Y) / N                    # gradient of loss w.r.t. z2
    dW2 = a1.T @ dz2                         # (64, 10)
    db2 = dz2.sum(axis=0)                    # (10,)

    da1 = dz2 @ W2.T                         # (1000, 64)
    dz1 = da1 * (z1 > 0)                     # ReLU derivative
    dW1 = X.T @ dz1                          # (784, 64)
    db1 = dz1.sum(axis=0)                    # (64,)

    # --- Update ---
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1
    return loss, probs

# Train for 200 epochs, recording accuracy each epoch
losses, accuracies = [], []
for epoch in range(200):
    loss, probs = train_epoch(X, Y, W1, b1, W2, b2, lr=0.1)
    losses.append(loss)
    accuracies.append((probs.argmax(axis=1) == y).mean())

print(f"Epoch   0: loss={losses[0]:.4f}  acc={accuracies[0]:.4f}")
print(f"Epoch  49: loss={losses[49]:.4f}  acc={accuracies[49]:.4f}")
print(f"Epoch 199: loss={losses[199]:.4f}  acc={accuracies[199]:.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.title("Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.subplot(1, 2, 2)
plt.plot(accuracies)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.tight_layout()
plt.savefig("mnist_training_curve.png", dpi=100)
plt.show()
print("Saved mnist_training_curve.png")


**What to notice:**
1. Loss drops from ~2.38 (≈ -log(0.1), random) to ~0.30 — the network is becoming confident and correct.
2. Accuracy climbs from ~0.06 to ~0.94 — a from-scratch numpy net reaches ~94% on MNIST.
3. The curve is steepest in the first ~25 epochs, then flattens — most learning happens early.

> **Pitfall:** Using a learning rate that is too large (e.g., 1.0) causes the loss to oscillate or diverge to NaN. Too small (e.g., 0.001) and 200 epochs won't be enough. 0.1 is a sweet spot for this setup.

---

## Exercises (do these!)

### Exercise 1 — 200-Epoch Training Curve
Run the training loop above and plot both loss and accuracy over 200 epochs. Report the final accuracy and the epoch where accuracy first exceeds 0.90.
*Expected: final accuracy ≈ 0.937; accuracy first exceeds 0.90 around epoch 128.*

**Follow-up:** What is the loss value at epoch 0, and why is it close to 2.30? Check: loss[0] ≈ 2.376.

### Exercise 2 — Learning Rate Showdown
Train two identical networks (same seed) with `lr=0.1` and `lr=0.01` for 200 epochs each. Plot both accuracy curves on the same axes. Report the final accuracy for each.
*Expected: lr=0.1 → ≈ 0.937; lr=0.01 → ≈ 0.651.*

**Follow-up:** At epoch 50, what is the accuracy gap between the two? Check: lr=0.1 ≈ 0.812 vs lr=0.01 ≈ 0.271.

### Exercise 3 — Confusion Matrix: 3 vs 8
Digits 3 and 8 are the most commonly confused pair. Train the network, then build a 2×2 confusion matrix using only the test rows whose true label is 3 or 8. Report the matrix and the accuracy on this subset.
*Expected: matrix [[90, 0], [8, 85]] (rows = true [3, 8], cols = predicted [3, 8]); subset accuracy ≈ 0.956.*

**Follow-up:** How many true 8s were misclassified as 3? Check: 8.

<details>
<summary>Hint</summary>

For Exercise 3, filter with `mask = (y == 3) | (y == 8)`, then loop over `(true, pred)` pairs and increment `cm[int(true==8), int(pred==8)]`.

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("mnist_train.csv", nrows=1000)
X = df.iloc[:, 1:].to_numpy(dtype=float) / 255.0
y = df["label"].to_numpy()
Y = np.eye(10)[y]

rng = np.random.default_rng(42)
W1 = rng.normal(0, 0.1, size=(784, 64)); b1 = np.zeros(64)
W2 = rng.normal(0, 0.1, size=(64, 10)); b2 = np.zeros(10)

def relu(z): return np.maximum(z, 0)
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

losses, accuracies = [], []
for epoch in range(200):
    z1 = X @ W1 + b1; a1 = relu(z1)
    probs = softmax(a1 @ W2 + b2)
    N = len(X)
    loss = -np.log(probs[np.arange(N), y] + 1e-12).mean()
    dz2 = (probs - Y) / N
    dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0)
    dz1 = (dz2 @ W2.T) * (z1 > 0)
    W2 -= 0.1 * dW2; b2 -= 0.1 * db2
    W1 -= 0.1 * (X.T @ dz1); b1 -= 0.1 * dz1.sum(axis=0)
    losses.append(loss)
    accuracies.append((probs.argmax(axis=1) == y).mean())

first_90 = next(i for i, a in enumerate(accuracies) if a > 0.90)
print(f"Final accuracy: {accuracies[-1]:.4f}")
print(f"First epoch > 0.90: {first_90}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(losses); plt.title("Loss"); plt.xlabel("Epoch")
plt.subplot(1, 2, 2); plt.plot(accuracies); plt.title("Accuracy"); plt.xlabel("Epoch")
plt.tight_layout(); plt.savefig("mnist_training_curve.png", dpi=100); plt.show()

# --- Solution 2 ---
def train(lr, epochs=200):
    rng = np.random.default_rng(42)
    W1 = rng.normal(0, 0.1, (784, 64)); b1 = np.zeros(64)
    W2 = rng.normal(0, 0.1, (64, 10)); b2 = np.zeros(10)
    accs = []
    for ep in range(epochs):
        z1 = X @ W1 + b1; a1 = relu(z1)
        probs = softmax(a1 @ W2 + b2)
        N = len(X)
        dz2 = (probs - Y) / N
        dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0)
        dz1 = (dz2 @ W2.T) * (z1 > 0)
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * (X.T @ dz1); b1 -= lr * dz1.sum(axis=0)
        accs.append((probs.argmax(axis=1) == y).mean())
    return accs

acc_01 = train(0.1)
acc_001 = train(0.01)
print(f"lr=0.1  final: {acc_01[-1]:.4f}")
print(f"lr=0.01 final: {acc_001[-1]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(acc_01, label="lr=0.1")
plt.plot(acc_001, label="lr=0.01")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.title("Learning Rate Comparison")
plt.savefig("lr_comparison.png", dpi=100); plt.show()

# --- Solution 3 ---
rng = np.random.default_rng(42)
W1 = rng.normal(0, 0.1, (784, 64)); b1 = np.zeros(64)
W2 = rng.normal(0, 0.1, (64, 10)); b2 = np.zeros(10)
for ep in range(200):
    z1 = X @ W1 + b1; a1 = relu(z1)
    probs = softmax(a1 @ W2 + b2)
    N = len(X)
    dz2 = (probs - Y) / N
    dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0)
    dz1 = (dz2 @ W2.T) * (z1 > 0)
    W2 -= 0.1 * dW2; b2 -= 0.1 * db2
    W1 -= 0.1 * (X.T @ dz1); b1 -= 0.1 * dz1.sum(axis=0)

pred = probs.argmax(axis=1)
mask = (y == 3) | (y == 8)
sub_y, sub_p = y[mask], pred[mask]
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(sub_y, sub_p):
    cm[int(t == 8), int(p == 8)] += 1
print("Confusion matrix (rows=true [3,8], cols=pred [3,8]):")
print(cm)
print(f"Subset accuracy: {np.trace(cm) / cm.sum():.4f}")

# --- Follow-up 1 ---
assert abs(losses[0] - 2.376) < 0.01, f"loss[0]={losses[0]}"
print(f"PASS: loss[0] = {losses[0]:.4f} ≈ -log(0.1) = 2.303 (random-guess loss)")

# --- Follow-up 2 ---
assert abs(acc_01[49] - acc_001[49]) > 0.5
print(f"PASS: epoch-50 gap = {acc_01[49]:.3f} vs {acc_001[49]:.3f}")

# --- Follow-up 3 ---
assert cm[1, 0] == 8
print(f"PASS: {cm[1, 0]} true 8s misclassified as 3")


### What to learn next
- Add a validation split and implement early stopping to prevent overfitting.
- Implement mini-batch gradient descent (update weights every 32 samples instead of the full batch).
- Add a second hidden layer and observe how depth changes the learning curve.
- Move to sklearn's `MLPClassifier` and compare its accuracy to your from-scratch net.
- Explore convolutional layers (CNNs), which exploit spatial structure in images and reach >99% on MNIST.

*Files in this folder: mnist_train.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
